# Nutrition Agent using Groq

This notebook sets up an AI-powered Nutrition Agent that analyzes food intake, estimates calories, and provides personalized nutritional advice using the Groq API.

In [1]:
!pip install groq -q

Now we will securely load your Groq API key from Google Colab's secrets manager and initialize the Groq client.

In [7]:
import os
from google.colab import userdata
from groq import Groq

# Retrieve Groq API key from Colab secrets
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    print("Please add 'GROQ_API_KEY' to your Colab secrets (the key icon on the left) before continuing.")
    GROQ_API_KEY = None

if GROQ_API_KEY:
    client = Groq(api_key=GROQ_API_KEY)
    print("Groq client successfully initialized!")

Groq client successfully initialized!


Let's create the Nutrition Agent class to handle analyzing meals and providing nutritional breakdowns.

In [9]:
class NutritionAgent:
    def __init__(self, client, model="openai/gpt-oss-120b"):
        self.client = client
        self.model = model

    def analyze_meal(self, meal_description):
        system_prompt = (
            "You are an expert, friendly AI Nutrition Agent. Your goal is to analyze the user's "
            "described meals, estimate the calories, break down macronutrients (carbohydrates, proteins, fats), "
            "and provide actionable health/dietary advice based on the description."
        )

        user_prompt = f"Analyze the following meal: {meal_description}"

        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.2
            )
            return completion.choices[0].message.content
        except Exception as e:
            return f"An error occurred: {e}"

# Instantiate the agent securely checking if client is defined
if 'client' in globals():
    agent = NutritionAgent(client=client)
    print("Nutrition Agent is ready!")
else:
    agent = None
    print("Error: 'client' is not defined. Please add your 'GROQ_API_KEY' to the secrets menu, then run the initialization cell above first.")

Nutrition Agent is ready!


Let's test the nutrition agent with a sample meal.

In [10]:
sample_meal = "2 scrambled eggs, 1 slice of whole-wheat toast with a half avocado, and a cup of black coffee"

if agent is not None:
    analysis = agent.analyze_meal(sample_meal)
    print("Analysis Output:\n")
    print(analysis)
else:
    print("Cannot test. The Nutrition Agent has not been successfully initialized because the Groq client is missing.")

Analysis Output:

**Meal Overview**

| Item | Approx. Portion | Calories | Carbs | Protein | Fat |
|------|----------------|----------|-------|---------|-----|
| Scrambled eggs (2 large) | 2 × ≈ 50 g each | ~140 kcal | 2 g | 12 g | 10 g |
| Whole‑wheat toast (1 slice) | ~30 g | ~70 kcal | 12 g | 3 g | 1 g |
| Avocado (½ medium) | ~70 g | ~110 kcal | 6 g | 1.5 g | 10 g |
| Black coffee (8 fl oz) | 0 kcal (no sugar/cream) | 0 g | 0 g | 0 g | 0 g |
| **Total** | — | **≈ 320 kcal** | **≈ 20 g** | **≈ 16.5 g** | **≈ 21 g** |

*Numbers are rounded averages from USDA FoodData Central. Real values can shift a bit depending on egg size, bread brand, and avocado ripeness.*

---

## What the Numbers Mean

| Nutrient | Why It Matters | How This Meal Stacks Up |
|----------|----------------|--------------------------|
| **Calories** | Energy balance (weight maintenance, loss, or gain). | ~320 kcal is a modest‑size breakfast, roughly 15‑20 % of a typical 2,000 kcal daily intake. |
| **Carbohydrates*